# Geospatial Anomaly Detection - Demo Notebook

This notebook demonstrates the full pipeline for detecting anomalous behaviors in geospatial trajectory data using deep learning and classical ML approaches.

**Pipeline Overview:**
1. Generate synthetic vessel/aircraft/vehicle trajectory data
2. Inject realistic anomalies (route deviations, speed anomalies, loitering, dark periods)
3. Extract spatial/temporal/kinematic features
4. Train and evaluate three detection models
5. Visualize results on interactive maps

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from src.data_generation import TrajectoryGenerator, AnomalyInjector
from src.feature_engineering import TrajectoryFeatureExtractor
from src.models import AutoencoderDetector, IsolationForestDetector, LSTMDetector
from src.visualization import TrajectoryMapVisualizer, EvaluationPlotter
from src.evaluation import AnomalyEvaluator

print('All modules loaded successfully.')

## 1. Synthetic Data Generation

We generate 300 trajectory tracks simulating vessel movements through the Strait of Malacca, aircraft along air corridors, and ground vehicles. The generator interpolates between waypoints with realistic speed and heading noise.

In [ ]:
# Generate base trajectories
generator = TrajectoryGenerator(seed=42, time_step_seconds=60)
raw_df = generator.generate_dataset(n_tracks=300, start_time='2025-01-01')

print(f'Generated {raw_df["track_id"].nunique()} tracks')
print(f'Total observations: {len(raw_df):,}')
print(f'\nEntity type distribution:')
print(raw_df.groupby('entity_type')['track_id'].nunique())
print(f'\nSample data:')
raw_df.head(10)

## 2. Anomaly Injection

We inject four types of anomalies into 15% of tracks:
- **Route Deviation**: Vessel/aircraft goes off-course
- **Speed Anomaly**: Sudden acceleration or near-stop
- **Loitering**: Circling pattern in an unusual area
- **Dark Period**: Gap in reporting (e.g., AIS transponder shutoff)

In [ ]:
# Inject anomalies
injector = AnomalyInjector(anomaly_fraction=0.15, seed=42)
df = injector.inject(raw_df)

n_anomalous = df[df['is_anomaly']]['track_id'].nunique()
n_total = df['track_id'].nunique()
print(f'Anomalous tracks: {n_anomalous}/{n_total} ({100*n_anomalous/n_total:.1f}%)')
print(f'Anomalous observations: {df["is_anomaly"].sum():,}/{len(df):,}')
print(f'\nAnomaly type distribution:')
print(df[df['anomaly_type'] != 'normal']['anomaly_type'].value_counts())

## 3. Interactive Map Visualization

Visualize trajectory tracks on an interactive map with anomalous segments highlighted in red.

In [ ]:
# Create interactive trajectory map
viz = TrajectoryMapVisualizer()
track_map = viz.create_track_map(df, max_tracks=40, show_anomalies=True)
track_map

In [ ]:
# Anomaly density heatmap
heatmap = viz.create_anomaly_heatmap(df)
heatmap

## 4. Feature Engineering

Extract per-point features (kinematics, temporal encoding, POI proximity) and per-track summary statistics.

In [ ]:
# Extract point-level features
extractor = TrajectoryFeatureExtractor()
point_df = extractor.extract_point_features(df)

print(f'Point-level features: {len(point_df.columns)} columns')
print(f'New features: acceleration, heading_change, cyclical time, POI distances, z-scores')

# Extract track-level features
track_df = extractor.extract_track_features(point_df)
print(f'\nTrack-level features: {len(track_df.columns)} columns')
track_df.head()

In [ ]:
# Feature distributions: normal vs anomalous
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
plot_features = [
    'speed_knots_mean', 'speed_knots_std', 'acceleration_std',
    'abs_heading_change_mean', 'total_distance_km', 'abs_heading_change_max'
]

for ax, feat in zip(axes.flat, plot_features):
    normal = track_df[~track_df['is_anomalous']][feat]
    anomalous = track_df[track_df['is_anomalous']][feat]
    ax.hist(normal, bins=25, alpha=0.6, label='Normal', color='#2196F3', density=True)
    ax.hist(anomalous, bins=25, alpha=0.6, label='Anomalous', color='#F44336', density=True)
    ax.set_title(feat.replace('_', ' ').title(), fontsize=10)
    ax.legend(fontsize=8)

fig.suptitle('Feature Distributions: Normal vs Anomalous Tracks', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Model Training & Evaluation

We train three models and compare their performance:
1. **Autoencoder** (PyTorch) - Learns normal pattern reconstruction
2. **Isolation Forest** (scikit-learn) - Baseline ensemble method
3. **LSTM** (PyTorch) - Sequence-aware deep learning

In [ ]:
# Prepare track-level features for Autoencoder and Isolation Forest
feature_cols = [c for c in track_df.columns if c not in 
                ('track_id', 'entity_type', 'is_anomalous', 'n_points', 'duration_hours')]
X = track_df[feature_cols].values.astype(np.float32)
y = track_df['is_anomalous'].astype(int).values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Train on normal data only for autoencoder
X_train_normal = X_train[y_train == 0]

print(f'Train set: {len(X_train)} ({y_train.sum()} anomalous)')
print(f'Test set: {len(X_test)} ({y_test.sum()} anomalous)')
print(f'Features: {X.shape[1]}')

In [ ]:
# --- Model 1: Autoencoder ---
print('Training Autoencoder...')
ae_detector = AutoencoderDetector(latent_dim=8, epochs=50, batch_size=64)
ae_detector.fit(X_train_normal)
ae_preds = ae_detector.predict(X_test)
ae_scores = ae_detector.score_samples(X_test)
print(f'Autoencoder predictions: {ae_preds.sum()} anomalies detected')

In [ ]:
# --- Model 2: Isolation Forest ---
print('Training Isolation Forest...')
if_detector = IsolationForestDetector(contamination=0.15, n_estimators=200)
if_detector.fit(X_train)
if_preds = if_detector.predict(X_test)
if_scores = if_detector.score_samples(X_test)
print(f'Isolation Forest predictions: {if_preds.sum()} anomalies detected')

In [ ]:
# --- Model 3: LSTM ---
print('Preparing sequence data for LSTM...')
seq_features = [
    'speed_knots', 'acceleration', 'heading_change',
    'abs_heading_change', 'hour_sin', 'hour_cos',
    'latitude', 'longitude',
]
X_seq, y_seq, seq_tids = extractor.get_sequence_data(
    point_df, feature_cols=seq_features, seq_length=30
)

# Split sequences
X_seq_train, X_seq_test, y_seq_train, y_seq_test = train_test_split(
    X_seq, y_seq, test_size=0.3, random_state=42, stratify=y_seq
)

print(f'Sequence data: {X_seq.shape}')
print(f'Anomalous sequences: {y_seq.sum()}/{len(y_seq)} ({100*y_seq.mean():.1f}%)')

print('\nTraining LSTM...')
lstm_detector = LSTMDetector(hidden_dim=64, n_layers=2, epochs=30, batch_size=128)
lstm_detector.fit(X_seq_train, y_seq_train)
lstm_preds = lstm_detector.predict(X_seq_test)
lstm_scores = lstm_detector.score_samples(X_seq_test)
print(f'LSTM predictions: {lstm_preds.sum()} anomalies detected')

## 6. Model Comparison

Compare all three models using ROC-AUC, precision-recall, and confusion matrices.

In [ ]:
# Evaluate all models
evaluator = AnomalyEvaluator()

ae_metrics = evaluator.evaluate(y_test, ae_preds, ae_scores, 'Autoencoder')
if_metrics = evaluator.evaluate(y_test, if_preds, if_scores, 'Isolation Forest')
lstm_metrics = evaluator.evaluate(y_seq_test, lstm_preds, lstm_scores, 'LSTM')

# Comparison table
comparison = evaluator.compare_models({
    'Autoencoder': ae_metrics,
    'Isolation Forest': if_metrics,
    'LSTM': lstm_metrics,
})

print('Model Comparison:')
comparison[['accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'avg_precision']]

In [ ]:
# Classification reports
for name, yt, yp in [('Autoencoder', y_test, ae_preds), 
                      ('Isolation Forest', y_test, if_preds),
                      ('LSTM', y_seq_test, lstm_preds)]:
    print(evaluator.print_report(yt, yp, name))

In [ ]:
# ROC Curves
plotter = EvaluationPlotter()

# Track-level models share the same test set
roc_results = {
    'Autoencoder': (y_test, ae_scores),
    'Isolation Forest': (y_test, if_scores),
    'LSTM': (y_seq_test, lstm_scores),
}

fig_roc = plotter.plot_roc_curves(roc_results)
plt.show()

In [ ]:
# Precision-Recall Curves
fig_pr = plotter.plot_precision_recall(roc_results)
plt.show()

In [ ]:
# Confusion Matrices
cm_results = {
    'Autoencoder': (y_test, ae_preds),
    'Isolation Forest': (y_test, if_preds),
    'LSTM': (y_seq_test, lstm_preds),
}
fig_cm = plotter.plot_confusion_matrices(cm_results)
plt.show()

In [ ]:
# Interactive Plotly comparison
fig_interactive = plotter.plot_interactive_comparison(roc_results)
fig_interactive.show()

## 7. Summary

### Key Findings

- **Autoencoder**: Learns the manifold of normal trajectory patterns; effective at detecting deviations in the feature space, particularly route deviations and loitering
- **Isolation Forest**: Strong baseline that works well on tabular track-level features without requiring neural network training
- **LSTM**: Captures temporal dependencies in trajectory sequences; best at detecting sequential anomalies like gradual speed changes and dark periods

### Applications

This framework is applicable to:
- **Maritime domain awareness**: Detecting vessels deviating from shipping lanes, unauthorized anchorage, AIS manipulation
- **Airspace monitoring**: Identifying non-conforming flight paths, unusual holding patterns
- **Border security**: Flagging unusual vehicle movement patterns near sensitive areas
- **Intelligence analysis**: Correlating movement anomalies across entities and time periods